# Feature Extraction **with** Scale Calibration

This notebook extracts cattle morphometric features **in centimetres (cm)** by applying per-image scale factors derived from:
- The **side (lateral)** view sticker (known physical size: 4 inches = 10.16 cm) → `S_side`
- The **withers height** bridge between views → `S_back`

### Calibration Logic

| Symbol | Derivation |
|--------|------------|
| `S_side` (cm/px) | `10.16 / √(sticker_mask_area_px)` |
| `WH_side_px` | `y_max - y_min` of side cow mask |
| `WH_back_px` | `y_max - y_min` of back cow mask |
| `S_back` (cm/px) | `S_side × (WH_side_px / WH_back_px)` |

### Extracted Features (in cm)
| Feature | Formula |
|---------|----------|
| Body Length (BL) | `S_side × BL_px` |
| Withers Height (WH) | `S_side × WH_side_px` |
| Chest Girth (CG) | `Ramanujan(a_cm, b_cm) × 1.15` where `a_cm = S_back × a_px`, `b_cm = S_side × b_px` |

In [7]:
import os
import sys
import cv2
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import time
import gc
from datetime import datetime, timedelta
from ultralytics import YOLO

print('Libraries imported successfully.')

Libraries imported successfully.


In [8]:
# ─── Configuration ────────────────────────────────────────────────────────────
STICKER_REAL_SIDE_CM = 4.0 * 2.54   # 10.16 cm
CHEST_X_RATIO        = 0.30

BASE_DIR        = r'c:\Users\bimyu\Documents\Projects\Oneject\playground'
DATA_DIR        = os.path.join(BASE_DIR, 'datasets', 'acme_ai', 'Pixel_B2', 'B2')
SEG_RESULT_DIR  = os.path.join(BASE_DIR, 'datasets', 'acme_ai', 'tests', 'segmentation_result')
FEAT_RESULT_DIR = os.path.join(BASE_DIR, 'datasets')
YOLO_STICKER    = os.path.join(BASE_DIR, 'models', 'best_sticker.pt')

SIDE_IMG_DIR = os.path.join(DATA_DIR, 'Side', 'images')
BACK_IMG_DIR = os.path.join(DATA_DIR, 'Back', 'images')

print(f'Sticker real side  : {STICKER_REAL_SIDE_CM} cm')
print(f'Sticker YOLO model : {YOLO_STICKER}')
print(f'Model available    : {os.path.exists(YOLO_STICKER)}')

Sticker real side  : 10.16 cm
Sticker YOLO model : c:\Users\bimyu\Documents\Projects\Oneject\playground\runs\segment\train_sticker\weights\best.pt
Model available    : True


In [9]:
# ─── Load sticker YOLO model ──────────────────────────────────────────────────
sticker_model = None
if os.path.exists(YOLO_STICKER):
    sticker_model = YOLO(YOLO_STICKER)
    print('Trained sticker YOLO model loaded — will use as primary detector.')
else:
    print('Trained sticker YOLO not found — will fall back to HSV+geometry.')

def detect_sticker_yolo(img):
    """
    Detect sticker mask using fine-tuned YOLO model.
    Uses results.masks.xy to draw smooth polygon at original resolution
    to avoid pixelated/blocky boundaries.
    """
    if sticker_model is None:
        return None
    results = sticker_model(img, retina_masks=True, verbose=False)[0]
    if results.masks is None or len(results.boxes) == 0:
        return None
    best = int(np.argmax(results.boxes.conf.cpu().numpy()))
    polygon_pts = results.masks.xy[best].astype(np.int32)
    mask = np.zeros(img.shape[:2], np.uint8)
    if len(polygon_pts) >= 3:
        cv2.fillPoly(mask, [polygon_pts], 255)
    return mask

def detect_sticker_hsv(img, cow_mask):
    """Multi-target HSV + square geometry fallback sticker detector."""
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    targets = [
        (np.array([25, 20,  80]), np.array([ 75, 255, 255])),
        (np.array([ 0,  0, 180]), np.array([180,  45, 255])),
        (np.array([15, 30,  80]), np.array([ 28, 255, 255])),
        (np.array([30, 30,  30]), np.array([ 75, 255, 180])),
    ]
    broad = np.zeros(img.shape[:2], np.uint8)
    for lo, hi in targets:
        broad = cv2.bitwise_or(broad, cv2.inRange(hsv, lo, hi))
    cands = cv2.bitwise_and(broad, cow_mask)
    k = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    cands = cv2.morphologyEx(cv2.morphologyEx(cands, cv2.MORPH_CLOSE, k), cv2.MORPH_OPEN, k)
    contours, _ = cv2.findContours(cands, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if not (200 <= area <= 10000):
            continue
        x, y, w, h = cv2.boundingRect(cnt)
        ar, ext = w/h, area/(w*h)
        if 0.6 <= ar <= 1.6 and ext >= 0.45:
            valid.append((abs(1-ar)+abs(0.9-ext), cnt))
    if not valid:
        return None
    mask = np.zeros(img.shape[:2], np.uint8)
    cv2.drawContours(mask, [min(valid, key=lambda x: x[0])[1]], -1, 255, -1)
    return mask

def get_sticker_scale(img, cow_mask_bin):
    """Return S_side (cm/px) from sticker, or None if undetected."""
    mask = None
    if sticker_model:
        mask = detect_sticker_yolo(img)
    if mask is None:
        mask = detect_sticker_hsv(img, cow_mask_bin)
    if mask is None:
        return None
    area = np.count_nonzero(mask)
    if area <= 0:
        return None
    return STICKER_REAL_SIDE_CM / np.sqrt(float(area))

print('Sticker detection functions ready.')


Trained sticker YOLO model loaded — will use as primary detector.
Sticker detection functions ready.


In [10]:
# ─── Morphometry helpers ──────────────────────────────────────────────────────

def extract_lateral(mask):
    """Return (y_min, y_max, BL_px, b_px) from grayscale side mask.
    b_px is the chest half-depth, clipped to the upper body (top 55% of
    mask height) to exclude leg pixels below the belly.
    """
    if mask is None:
        return None
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    wy, wx = np.where(mask == 255)
    if len(wy) == 0:
        return None
    y_min, y_max = int(wy.min()), int(wy.max())
    x_min, x_max = int(wx.min()), int(wx.max())
    BL_px = float(x_max - x_min)
    chest_x = x_min + CHEST_X_RATIO * BL_px
    # Clip to upper body only: exclude leg pixels below 55% of mask height
    body_cutoff = int(y_min + 0.55 * (y_max - y_min))
    col_full = np.where(mask[:, int(chest_x)] == 255)[0]
    col = col_full[col_full <= body_cutoff]  # keep only body (above belly)
    if len(col) == 0:
        col = col_full  # fallback to full column if clip removed everything
    b_px = float(col.max() - col.min()) / 2.0 if len(col) > 0 else 0.0
    return y_min, y_max, BL_px, b_px

def extract_back(mask):
    """Return (y_min, y_max, a_px, x_center, y_chest) from grayscale back mask.
    Chest half-width (a) is measured at 25% from mask top (upper trunk),
    which positions the ellipse on the main body rather than the thigh region.
    """
    if mask is None:
        return None
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    wy, wx = np.where(mask == 255)
    if len(wy) == 0:
        return None
    y_min, y_max = int(wy.min()), int(wy.max())
    x_min, x_max = int(wx.min()), int(wx.max())
    width_px = float(x_max - x_min)
    # Chest horizontal half-width (a) at 25% down from mask top (upper trunk)
    y_chest = y_min + 0.25 * (y_max - y_min)
    row = np.where(mask[int(y_chest), :] == 255)[0]
    if len(row) > 0:
        a_px = float(row.max() - row.min()) / 2.0
        x_center = float(row.min() + row.max()) / 2.0
    else:
        a_px = width_px / 4.0
        x_center = float(x_min + x_max) / 2.0
    return y_min, y_max, a_px, x_center, float(y_chest)

def ramanujan_girth(a, b, correction=1.15):
    """Ramanujan ellipse circumference with correction factor."""
    if a <= 0 or b <= 0:
        return 0.0
    h = ((a - b) ** 2) / ((a + b) ** 2)
    return correction * np.pi * (a + b) * (1.0 + (3.0 * h) / (10.0 + np.sqrt(4.0 - 3.0 * h)))

print('Morphometry helpers defined.')

Morphometry helpers defined.


In [11]:
# ─── Align side ↔ back image pairs ───────────────────────────────────────────
matched_data = []
back_index = {}  # cow_id → back filename
for bf in os.listdir(BACK_IMG_DIR):
    if bf.lower().endswith(('.jpg', '.jpeg', '.png')):
        try:
            back_index[int(float(bf.split('_')[0]))] = bf
        except ValueError:
            pass

for f in os.listdir(SIDE_IMG_DIR):
    if not f.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    parts = f.split('_')
    if len(parts) < 3:
        continue
    try:
        idx = int(float(parts[0]))
        weight_kg = float(parts[2])
    except ValueError:
        continue
    if idx not in back_index:
        continue
    matched_data.append({
        'id'        : idx,
        'weight_kg' : weight_kg,
        'side_img'  : os.path.join(SIDE_IMG_DIR, f),
        'back_img'  : os.path.join(BACK_IMG_DIR, back_index[idx]),
    })

matched_data = sorted(matched_data, key=lambda x: x['id'])
print(f'Successfully aligned {len(matched_data)} matched cow pairs.')

Successfully aligned 509 matched cow pairs.


In [12]:
# ─── Main extraction loop (pixels → cm) ──────────────────────────────────────
models_list  = ['yolov8l']
features_by_model = {m: {} for m in models_list}
# Also track raw scale factors per cow
scale_factors = {}  # cow_id → {S_side, S_back}

for m in models_list:
    os.makedirs(os.path.join(FEAT_RESULT_DIR, m), exist_ok=True)

print('Starting calibrated feature extraction...')
start_time = time.time()

for count, sample in enumerate(matched_data):
    idx = sample['id']
    ts  = datetime.now().strftime('%H:%M:%S')
    if count >= 1:
        elapsed = time.time() - start_time
        eta = datetime.now() + timedelta(seconds=(elapsed/count) * (len(matched_data)-count))
        print(f'[{ts}] Cow {idx} ({count+1}/{len(matched_data)}) ETA {eta.strftime("%H:%M:%S")}')
    else:
        print(f'[{ts}] Cow {idx} (1/{len(matched_data)})')

    lat_img = cv2.imread(sample['side_img'])
    top_img = cv2.imread(sample['back_img'])
    if lat_img is None or top_img is None:
        continue

    lat_fname = os.path.splitext(os.path.basename(sample['side_img']))[0] + '.png'
    top_fname = os.path.splitext(os.path.basename(sample['back_img']))[0] + '.png'

    # ── Compute S_side from the sticker in the side image ──────────────────
    # We use the yolov8l side mask as a quick cow region for the HSV fallback
    lat_mask_l = cv2.imread(os.path.join(SEG_RESULT_DIR, 'yolov8l', lat_fname), cv2.IMREAD_GRAYSCALE)
    cow_bin = lat_mask_l if lat_mask_l is not None else np.zeros(lat_img.shape[:2], np.uint8)

    S_side = get_sticker_scale(lat_img, cow_bin)

    # ── Per-model feature extraction ────────────────────────────────────────
    for m_name in models_list:
        lat_mask = cv2.imread(os.path.join(SEG_RESULT_DIR, m_name, lat_fname), cv2.IMREAD_GRAYSCALE)
        top_mask = cv2.imread(os.path.join(SEG_RESULT_DIR, m_name, top_fname), cv2.IMREAD_GRAYSCALE)
        if lat_mask is None or top_mask is None:
            continue

        try:
            lat_res = extract_lateral(lat_mask)
            top_res = extract_back(top_mask)
            if lat_res is None or top_res is None:
                continue

            y_min_lat, y_max_lat, BL_px, b_px = lat_res
            y_min_top, y_max_top, a_px, x_center_top, y_chest_top = top_res

            WH_side_px = float(y_max_lat - y_min_lat)   # withers height in side px
            WH_back_px = float(y_max_top - y_min_top)   # withers height in back px

            # ── S_back via withers height bridge ─────────────────────────
            if S_side is not None and WH_back_px > 0:
                S_back = S_side * (WH_side_px / WH_back_px)
            else:
                S_back = None

            # ── Convert to cm ─────────────────────────────────────────────
            if S_side is not None and S_back is not None:
                BL_cm = S_side * BL_px
                WH_cm = S_side * WH_side_px        # = S_back * WH_back_px
                b_cm  = S_side * b_px
                a_cm  = S_back * a_px
                CG_cm = ramanujan_girth(a_cm, b_cm)
            else:
                # fallback: store NaN (no scale factor available)
                BL_cm = WH_cm = CG_cm = np.nan
                S_back = np.nan

            # Store scale factors (from yolov8l model pass — same for all)
            if m_name == 'yolov8l' and idx not in scale_factors:
                scale_factors[idx] = {
                    'S_side': round(S_side, 6) if S_side else np.nan,
                    'S_back': round(S_back, 6) if S_back and not np.isnan(S_back) else np.nan,
                }

            features_by_model[m_name][idx] = {
                'BL'   : round(BL_cm, 4),
                'WH'   : round(WH_cm, 4),
                'CG'   : round(CG_cm, 4),
                'BL_px': round(BL_px, 1),
                'WH_px': round(WH_side_px, 1),
                'CG_px': round(ramanujan_girth(a_px, b_px), 1),
            }

            # ── Visualization ─────────────────────────────────────────────
            lat_rgb = cv2.cvtColor(lat_img, cv2.COLOR_BGR2RGB)
            top_rgb = cv2.cvtColor(top_img, cv2.COLOR_BGR2RGB)
            top_mask_rgb = cv2.cvtColor(top_mask, cv2.COLOR_GRAY2RGB)

            ov_lat = lat_rgb.copy(); ov_lat[lat_mask==255] = [0,255,0]
            ov_top = top_rgb.copy(); ov_top[top_mask==255] = [0,255,0]
            lat_b = cv2.addWeighted(lat_rgb, 0.8, ov_lat, 0.2, 0)
            top_b = cv2.addWeighted(top_rgb, 0.8, ov_top, 0.2, 0)

            wy_lat, wx_lat = np.where(lat_mask==255)
            x_min_lat = wx_lat.min()
            chest_x_lat = x_min_lat + CHEST_X_RATIO * BL_px
            col_lat = np.where(lat_mask[:, int(chest_x_lat)]==255)[0]
            ys_lat = col_lat.min() if len(col_lat)>0 else y_min_lat
            ye_lat = col_lat.max() if len(col_lat)>0 else y_max_lat

            fig, axes = plt.subplots(1, 3, figsize=(24, 7))

            axes[0].imshow(lat_b)
            axes[0].plot([x_min_lat+BL_px/2]*2, [y_min_lat, y_max_lat],
                         'c-', lw=4, label=f'WH = {WH_cm:.1f} cm')
            axes[0].plot([x_min_lat, x_min_lat+BL_px], [y_min_lat+200]*2,
                         'y-', lw=4, label=f'BL = {BL_cm:.1f} cm')
            axes[0].plot([chest_x_lat]*2, [ys_lat, ye_lat],
                         'm-', lw=4, label=f'2b = {2*b_cm:.1f} cm')
            axes[0].set_title(f'Cow {idx} Side · {m_name}\nS_side={S_side:.5f} cm/px')
            axes[0].legend(loc='upper right'); axes[0].axis('off')

            axes[1].imshow(top_b)
            axes[1].plot([x_center_top-a_px, x_center_top+a_px], [y_chest_top]*2,
                         'm-', lw=4, label=f'2a = {2*a_cm:.1f} cm')
            axes[1].set_title(f'Cow {idx} Back · {m_name}\nS_back={S_back:.5f} cm/px' if S_back and not np.isnan(S_back) else f'Cow {idx} Back · {m_name}')
            axes[1].legend(loc='upper right'); axes[1].axis('off')

            axes[2].imshow(top_mask_rgb)
            ellipse = patches.Ellipse(
                (x_center_top, y_chest_top), 2*a_px, 2*b_px,
                fill=True, color='lightblue', alpha=0.6, edgecolor='blue', lw=3)
            axes[2].add_patch(ellipse)
            axes[2].plot([x_center_top-a_px, x_center_top+a_px], [y_chest_top]*2,
                         'r--', lw=2, label=f'a={a_px:.0f}px')
            axes[2].plot([x_center_top]*2, [y_chest_top-b_px, y_chest_top+b_px],
                         'g--', lw=2, label=f'b={b_px:.0f}px')
            axes[2].set_title(f'CG = {CG_cm:.1f} cm')
            axes[2].legend(loc='upper right'); axes[2].axis('off')

            plt.tight_layout()
            plt.savefig(os.path.join(FEAT_RESULT_DIR, m_name, f'{idx}_calibrated.png'), bbox_inches='tight')
            plt.close(fig)

        except Exception as e:
            if 'fig' in dir():
                plt.close(fig)
            print(f'  [WARN] Cow {idx} {m_name}: {e}')
            continue

    if (count + 1) % 10 == 0:
        gc.collect()

print('\nExtraction complete.')

Starting calibrated feature extraction...
[11:40:00] Cow 1 (1/509)
[11:40:01] Cow 2 (2/509) ETA 11:53:50
[11:40:02] Cow 3 (3/509) ETA 11:49:24
[11:40:02] Cow 4 (4/509) ETA 11:48:11
[11:40:03] Cow 5 (5/509) ETA 11:47:55
[11:40:04] Cow 6 (6/509) ETA 11:47:46
[11:40:05] Cow 7 (7/509) ETA 11:47:40
[11:40:06] Cow 8 (8/509) ETA 11:47:30
[11:40:06] Cow 9 (9/509) ETA 11:47:06
[11:40:08] Cow 10 (10/509) ETA 11:47:58
[11:40:09] Cow 11 (11/509) ETA 11:48:21
[11:40:10] Cow 12 (12/509) ETA 11:48:15
[11:40:11] Cow 13 (13/509) ETA 11:48:08
[11:40:12] Cow 14 (14/509) ETA 11:47:51
[11:40:12] Cow 15 (15/509) ETA 11:47:46
[11:40:13] Cow 16 (16/509) ETA 11:47:45
[11:40:14] Cow 17 (17/509) ETA 11:47:46
[11:40:15] Cow 18 (18/509) ETA 11:47:51
[11:40:16] Cow 19 (19/509) ETA 11:47:43
[11:40:17] Cow 20 (20/509) ETA 11:47:42
[11:40:18] Cow 21 (21/509) ETA 11:47:46
[11:40:19] Cow 22 (22/509) ETA 11:47:47
[11:40:20] Cow 23 (23/509) ETA 11:47:52
[11:40:21] Cow 24 (24/509) ETA 11:47:44
[11:40:21] Cow 25 (25/509) ET

In [16]:
# ─── Compile and save features.csv ───────────────────────────────────────────
rows_csv = []
for sample in matched_data:
    idx = sample['id']
    sf  = scale_factors.get(idx, {'S_side': np.nan, 'S_back': np.nan})

    row = {
        'id'          : idx,
        'scale_factor': sf['S_side'],        # Column B
        'scale_factor_back': sf['S_back'],   # Column C
        'weight_kg'   : sample['weight_kg'],
    }

    for m_name in models_list:
        feat = features_by_model[m_name].get(idx,
               {'BL': np.nan, 'WH': np.nan, 'CG': np.nan,
                'BL_px': np.nan, 'WH_px': np.nan, 'CG_px': np.nan})
        row[f'BL_{m_name}']    = feat['BL']     # cm
        row[f'WH_{m_name}']    = feat['WH']     # cm
        row[f'CG_{m_name}']    = feat['CG']     # cm
        row[f'BL_px_{m_name}'] = feat['BL_px']  # pixels (kept for audit)
        row[f'WH_px_{m_name}'] = feat['WH_px']  # pixels
        row[f'CG_px_{m_name}'] = feat['CG_px']  # pixels

    rows_csv.append(row)

df_out = pd.DataFrame(rows_csv)

csv_path = os.path.join(FEAT_RESULT_DIR, 'features.csv')
df_out.to_csv(csv_path, index=False, sep=';')
print(f'Saved {len(df_out)} rows → {csv_path}')
df_out.head(10)

Saved 509 rows → c:\Users\bimyu\Documents\Projects\Oneject\playground\datasets\acme_ai\tests\feature_extraction_result\features.csv


,id,scale_factor,scale_factor_back,weight_kg,BL_yolov8l,WH_yolov8l,CG_yolov8l,BL_px_yolov8l,WH_px_yolov8l,CG_px_yolov8l
0,1,NaN,NaN,117.0,NaN,NaN,NaN,NaN,NaN,NaN
1,2,NaN,NaN,61.0,NaN,NaN,NaN,NaN,NaN,NaN
2,3,NaN,NaN,86.0,NaN,NaN,NaN,NaN,NaN,NaN
3,4,NaN,NaN,165.0,NaN,NaN,NaN,NaN,NaN,NaN
4,5,NaN,NaN,157.0,NaN,NaN,NaN,NaN,NaN,NaN
5,6,NaN,NaN,96.0,NaN,NaN,NaN,NaN,NaN,NaN
6,7,NaN,NaN,139.0,NaN,NaN,NaN,NaN,NaN,NaN
7,8,NaN,NaN,107.0,NaN,NaN,NaN,NaN,NaN,NaN
8,9,NaN,NaN,181.0,NaN,NaN,NaN,NaN,NaN,NaN
9,10,NaN,NaN,141.0,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# ─── Quick sanity check ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt
matplotlib.use('inline')  # switch to inline for display

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for i, m in enumerate(models_list):
    col = f'BL_{m}'
    axes[i].hist(df_out[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(f'{col}\nmean={df_out[col].mean():.1f} cm')
    axes[i].set_xlabel('cm')
plt.suptitle('Body Length Distribution (cm) per Model', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nScale factor summary:')
print(df_out[['scale_factor','scale_factor_back']].describe().round(6))